# Pre-Entrega 6: Orquestador Multi-Agente de Análisis e Investigación

> **Programa:** AI Engineering — Coderhouse  
> **Módulo 6:** Sistemas Multi-Agente: Colaboración y Especialización  
> **Topología:** Jerárquica con Patrón Supervisor en LangGraph  

---

## 🎯 Objetivo de la Demostración
Este notebook demuestra el funcionamiento en vivo del **Orquestador Multi-Agente**, ilustrando:
1. **Delegación Dinámica:** El nodo Supervisor rutea al `Investigador` para recopilar datos de mercado.
2. **Cómputo Especializado:** El Supervisor enruta al `Analista` para calcular métricas cuantitativas y CAGR.
3. **Compuerta de Calidad y Síntesis Final:** Validación de suficiencia y generación de informe ejecutivo consolidado.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

# Cargar variables de entorno
load_dotenv(Path(".").resolve() / ".env")
load_dotenv(Path("..").resolve() / "pre_entrega_05" / ".env")

from graph import graph
print("✅ Grafo del Orquestador importado y compilado con éxito.")

## 🚀 Ejecución del Flujo con Streaming Paso a Paso
Lanzamos una consulta compleja que requiere investigación factual y procesamiento matemático cuantitativo.

In [ ]:
user_query = (
    "Investiga las proyecciones de mercado y tasas de adopción de IA Generativa "
    "para el período 2024-2030 y calcula la tasa de crecimiento anual compuesta (CAGR)."
)

initial_state = {
    "messages": [HumanMessage(content=user_query)],
    "next_agent": "supervisor",
    "research_data": None,
    "analysis_data": None,
    "final_summary": None,
    "iteration_count": 0,
    "error": None
}

print(f"📥 Consulta: {user_query}\n")
print("=" * 75)

step_count = 1
for event in graph.stream(initial_state):
    for node_name, output in event.items():
        print(f"\n[Paso {step_count}] 🏷️ Nodo Activo: '{node_name}'")
        if "next_agent" in output:
            print(f"   👉 Decisión del Supervisor: next_agent -> '{output['next_agent']}'")
        if "messages" in output:
            for msg in output["messages"]:
                sender = getattr(msg, "name", "Sistema")
                print(f"\n💬 [{sender}]:\n{msg.content}\n")
        step_count += 1
        print("-" * 75)

## 📊 Inspección de Artefactos Estructurados en el Estado
Comprobamos cómo el estado compartido mantiene los canales de datos aislados y tipados con Pydantic V2.

In [ ]:
final_state = graph.invoke(initial_state)

print("=== 🔍 ARTEFACTO DE INVESTIGACIÓN (ResearchArtifact) ===")
print(final_state.get("research_data"))

print("\n=== 📈 ARTEFACTO DE ANÁLISIS (AnalysisArtifact) ===")
print(final_state.get("analysis_data"))

## 🏛️ Topología Visual del Grafo
Representación del grafo en formato Mermaid.

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Código Mermaid del grafo:\n")
    print(graph.get_graph().draw_mermaid())